In [1]:

import numpy as np
import pickle
from collections import Counter

from qiskit import transpile, QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.circuit.library import QAOAAnsatz
from qiskit.transpiler.passes.routing.commuting_2q_gate_routing import SwapStrategy

from qiskit_aer import AerSimulator
from qiskit_aer.primitives import SamplerV2 as Sampler
from qiskit_ibm_runtime.fake_provider import FakeFez

from qopt_best_practices.sat_mapping import SATMapper

from qiskit_qaoa.utils.circuit_graph_utils import circuit_to_graph, graph_to_operator, circuit_construction
from qiskit_qaoa.utils.hamiltonian_utils import get_Q_and_hamiltonian
from qiskit_qaoa.utils.string_utils import evaluate_sparse_pauli_samples, evaluate_sparse_pauli


In [2]:
backend_options = dict(
    # method='matrix_product_state',
    method='statevector',
    matrix_product_state_max_bond_dimension='20', 
    device='CPU',
    precision='single',
    basis_gates = ['rx', 'ry', 'rz', 'cx']
)
# fake_fez = FakeFez()
backend = AerSimulator(**backend_options)

In [3]:
sampler = Sampler.from_backend(backend)

In [4]:
filename = 'test_N2_W2'
N = 2
p: int = 6
shots = 4000

rng = np.random.default_rng()

data_file = f'/lustre/scratch127/qpg/jc59/out/oriented/qubo_data_{filename}.gfa.pkl'

Q, hamiltonian, offset, ising_offset = get_Q_and_hamiltonian(data_file)
num_qubits: int = hamiltonian.num_qubits

In [5]:
delta_beta, delta_gamma = 0.17, 0.17
betas = [(1-k/p) * delta_beta for k in range(p)]
gammas = [(k+1) / p * delta_gamma for k in range(p)]
fixed_params = betas + gammas

In [6]:
phis = ParameterVector('ϕ', num_qubits)
betas = ParameterVector('β', p)


init = QuantumCircuit(num_qubits)
for i in range(num_qubits):
    init.ry(phis[i], i)
    
mixer = QuantumCircuit(num_qubits)
for i in range(num_qubits):
    mixer.ry(-phis[i], i)
    mixer.rz(-2*betas[0], i)
    mixer.ry(phis[i], i)
    
qc = QAOAAnsatz(
    cost_operator=hamiltonian,
    reps = p,
    flatten=True
)


In [61]:
u_backend = AerSimulator(method='unitary')
mixer_test = QuantumCircuit(1)
i = 0
c = 0.25
beta = np.pi/4
theta = 2*np.arcsin(np.sqrt(c))
mixer_test.ry(-theta, i)
mixer_test.rz(2*beta, i)
mixer_test.ry(theta, i)
mixer_test.save_unitary()
t_res = u_backend.run(mixer_test).result()

In [62]:
O = np.asarray(t_res.results[0].data.unitary)
v = np.array([
    [np.cos(theta/2), -np.sin(theta/2)],
    [np.sin(theta/2), np.cos(theta/2)]
]) @ np.array([[1], [0]])
print(O @ v)
print(v)
rat = O @ v / v
print(np.arctan(np.imag(rat[0])/np.real(rat[0]))[0], beta)

[[0.61237244-0.61237244j]
 [0.35355339-0.35355339j]]
[[0.8660254]
 [0.5      ]]
-0.7853981633974483 0.7853981633974483


In [63]:
graph = circuit_to_graph(qc, qc.parameters[p])

swap_strat = SwapStrategy.from_line(range(graph.order()))
edge_coloring = {(idx, idx + 1): (idx + 1) % 2 for idx in range(graph.order())}

remapped_g, sat_map, min_sat_layers = SATMapper(timeout=30).remap_graph_with_sat(
    graph=graph, swap_strategy=swap_strat
)
if remapped_g is None:
    raise Exception('Failed to find initial layout')

cost_op = graph_to_operator(remapped_g)
singles = cost_op[cost_op.paulis.z.sum(axis=-1) == 1]
doubles = cost_op[cost_op.paulis.z.sum(axis=-1) == 2]

circ_dict = circuit_construction(singles, doubles, None, swap_strat, edge_coloring, {}, p, init, mixer)

circuit = circ_dict["circuit_to_sample"]

In [64]:
circuit.parameters

ParameterView([ParameterVectorElement(β[0]), ParameterVectorElement(β[1]), ParameterVectorElement(β[2]), ParameterVectorElement(β[3]), ParameterVectorElement(β[4]), ParameterVectorElement(β[5]), ParameterVectorElement(γ[0]), ParameterVectorElement(γ[1]), ParameterVectorElement(γ[2]), ParameterVectorElement(γ[3]), ParameterVectorElement(γ[4]), ParameterVectorElement(γ[5]), ParameterVectorElement(ϕ[0]), ParameterVectorElement(ϕ[1]), ParameterVectorElement(ϕ[2]), ParameterVectorElement(ϕ[3]), ParameterVectorElement(ϕ[4]), ParameterVectorElement(ϕ[5]), ParameterVectorElement(ϕ[6]), ParameterVectorElement(ϕ[7]), ParameterVectorElement(ϕ[8]), ParameterVectorElement(ϕ[9])])

In [65]:
fixed_param_bind = {circuit.parameters[i]: fixed_params[i] for i in range(2*p)}
fixed_qc = circuit.assign_parameters(fixed_param_bind)

In [66]:
fixed_qc.draw(fold=-1)

┌──────────┐ ┌───────────┐                                              ┌───┐                                                         ┌───┐                                                         ┌───┐                                                         ┌───┐                                                                            ┌───────────┐┌───────────┐┌──────────┐                                                            ┌───┐                                                       ┌───┐                                                            ┌───┐                                                                      ┌───┐                      ┌──────────┐  ┌───────────┐ ┌──────────────┐  ┌──────────┐  ┌───────────┐                                                                                      ┌───┐                                                     ┌───┐                                                     ┌───┐                                                     ┌───┐                                                                           ┌───────────┐┌──────────────┐┌──────────┐                                                           ┌───┐                                                       ┌───┐                                                            ┌───┐                                                                     ┌───┐                     ┌──────────┐ ┌───────────┐ ┌───────────┐  ┌──────────┐  ┌───────────┐                                                                                    ┌───┐                                                       ┌───┐                                                       ┌───┐                                                       ┌───┐                                                                            ┌───────────┐┌──────────────┐┌──────────┐                                                      ┌───┐                                                   ┌───┐                                                       ┌───┐                                                               ┌───┐                   ┌──────────┐ ┌───────────┐┌───────────────┐ ┌──────────┐              ┌─┐                                                     
 q_0: ┤ Ry(ϕ[0]) ├─┤ Rz(-0.85) ├────────────────────────────■─────────────────┤ X ├──■───────────────────────────────────■──────────────────┤ X ├──■───────────────────────────────────■──────────────────┤ X ├──■───────────────────────────────────■──────────────────┤ X ├──■───────────────────────────────────────■──────────────────────────■──────┤ Ry(-ϕ[0]) ├┤ Rz(-0.34) ├┤ Ry(ϕ[0]) ├──■───────────────────■──────────────────────────────────■──┤ X ├─────────────────■──────────────────────────────────■──┤ X ├─────────────────■──────────────────────────────────■───────┤ X ├──────────────────────■────────────────────────────────────────────■──┤ X ├─────────────────■────┤ Rz(-1.7) ├──┤ Ry(-ϕ[0]) ├─┤ Rz(-0.28333) ├──┤ Ry(ϕ[0]) ├──┤ Rz(-2.55) ├─────────────────────────────────────────────────────────────────────■────────────────┤ X ├──■─────────────────────────────────■────────────────┤ X ├──■─────────────────────────────────■────────────────┤ X ├──■─────────────────────────────────■────────────────┤ X ├──■─────────────────────────────────────■───────────────────────────■──────┤ Ry(-ϕ[0]) ├┤ Rz(-0.22667) ├┤ Ry(ϕ[0]) ├──■──────────────────■──────────────────────────────────■──┤ X ├─────────────────■──────────────────────────────────■──┤ X ├─────────────────■──────────────────────────────────■───────┤ X ├──────────────────────■───────────────────────────────────────────■──┤ X ├────────────────■────┤ Rz(-3.4) ├─┤ Ry(-ϕ[0]) ├─┤ Rz(-0.17) ├──┤ Ry(ϕ[0]) ├──┤ Rz(-4.25) ├───────────────────────────────────────────────────────────────────■────────────────┤ X ├──■──────────────────────────────────■─────────────────┤ X ├──■──────────────────────────────────■─────────────────┤ X ├──■──────────────────────────────────■─────────────────┤ X ├──■─────

In [67]:
eta = 1
eps = 0.03

def normalisation(energies):
    return sum([np.exp(-beta_T * E) for E in energies])

def boltzmann(energies, Z):
    return np.exp(- beta_T * energies) / Z

def bias(boltzmanns, q, samples):
    return sum([boltzmanns[i] * (-1 if samples[i][q] == '1' else 1) for i in range(len(samples))])

def get_biases(samples):
    energies = evaluate_sparse_pauli_samples(samples, hamiltonian) + ising_offset
    Z = normalisation(energies)
    boltzmanns = boltzmann(energies, Z)
    return np.array([bias(boltzmanns, q, samples) for q in range(num_qubits)])

def get_angles(samples):
    biases = get_biases(samples)
    probabilities = 0.5 * (1 - eta * biases)
    probabilities[probabilities < eps] = eps
    probabilities[probabilities > 1- eps] = 1- eps
    angles = 2 * np.arcsin(np.sqrt(probabilities))
    return angles

In [68]:
def iteration(angles):
    sample_circuit = fixed_qc.assign_parameters(angles, inplace=False)
    sampler_job = sampler.run([sample_circuit],shots=shots)
    sampler_result = sampler_job.result()
    counts = sampler_result[0].data.c.get_counts()
    all_samples = [[sample] * count for sample, count in counts.items()]
    flat_all_samples = [x for xs in all_samples for x in xs]
    history.append(counts)
    new_angles = get_angles(flat_all_samples)
    return new_angles

In [76]:
# init_angles = np.pi/2 * np.ones((num_qubits,))
theta = 2*np.arcsin((2*N+1)**-0.5 )
init_angles = theta * np.ones((num_qubits,))
# init_angles[2] += 0.2
history = []
angles = [init_angles]
for i in range(4):
    beta_T = 0.1 * (i+1)
    angles.append(iteration(angles[-1]))

In [81]:
counter = Counter(history[2])
print(counter.most_common(10))
evaluate_sparse_pauli_samples([e[0] for e in counter.most_common(10)], hamiltonian) + ising_offset

[('1000000001', 647), ('1000000100', 352), ('0010000001', 296), ('0001000001', 265), ('0100000001', 244), ('1000001000', 148), ('1000000010', 138), ('0010000100', 126), ('1000000000', 114), ('0100000100', 103)]


array([ 1.,  1.,  0.,  7.,  5.,  1.,  1.,  7., 12.,  7.])

In [71]:
# 1000000 0000010 0001000 0100000